# Phase 1 — Diffusion Autoencoder Training
**Continuous Sign Language Translation** · Phase 1 of 2

Trains a `SemanticEncoder` (1D-CNN) + `DiffusionDecoder` (1D-UNet) on 100 clips of the How2Sign landmark dataset (streaming from Hugging Face).  
The encoder learns a compact latent `Z [B, 15, 512]` that will be used in Phase 2.

**Runtime:** GPU (T4 or better) · **Est. time (100 clips, 3 epochs):** ~2 min


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
%pip install -q torch datasets huggingface_hub tqdm numpy


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────────────
# Option A (recommended): Colab Secrets  →  Runtime > Manage secrets > add HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)


## Model definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [B, T, D]
        x = x + self.pe[:, :x.size(1), :]
        return x

class PartSpatialEncoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.GELU(),
            nn.LayerNorm(out_dim), nn.Linear(out_dim, out_dim),
        )
    def forward(self, x): return self.net(x)

class MultiStreamSemanticEncoder(nn.Module):
    def __init__(self, d_model=384, latent_dim=512):
        super().__init__()
        part_dim = d_model // 4
        self.body_encoder = PartSpatialEncoder(33*3*2, part_dim)
        self.face_encoder = PartSpatialEncoder(15*3*2, part_dim)
        self.lhand_encoder = PartSpatialEncoder(21*3*2, part_dim)
        self.rhand_encoder = PartSpatialEncoder(21*3*2, part_dim)
        self.fusion = nn.Sequential(nn.Linear(part_dim * 4, d_model), nn.GELU(), nn.LayerNorm(d_model))
        
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, activation="gelu",
        )
        self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.downsample = nn.Conv1d(d_model, latent_dim, kernel_size=3, stride=2, padding=1)
        self.downsample2 = nn.Conv1d(latent_dim, latent_dim, kernel_size=3, stride=2, padding=1)

    def forward(self, inputs, src_key_padding_mask=None):
        body = torch.cat([inputs["body_pos"], inputs["body_vel"]], dim=-1)
        face = torch.cat([inputs["face_pos"], inputs["face_vel"]], dim=-1)
        lhand = torch.cat([inputs["lhand_pos"], inputs["lhand_vel"]], dim=-1)
        rhand = torch.cat([inputs["rhand_pos"], inputs["rhand_vel"]], dim=-1)

        b_feat = self.body_encoder(body)
        f_feat = self.face_encoder(face)
        l_feat = self.lhand_encoder(lhand)
        r_feat = self.rhand_encoder(rhand)

        fused = self.fusion(torch.cat([b_feat, f_feat, l_feat, r_feat], dim=-1))
        fused = self.pos_encoder(fused)
        
        temp_out = self.temporal(fused, src_key_padding_mask=src_key_padding_mask)
        
        x = temp_out.transpose(1, 2)
        x = F.gelu(self.downsample(x))
        x = F.gelu(self.downsample2(x))
        return x.transpose(1, 2)

class StructuredDiffusionDecoder(nn.Module):
    def __init__(self, latent_dim=512):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(1, 128), nn.GELU(), nn.Linear(128, 128))
        self.upsample_z = nn.ConvTranspose1d(latent_dim, latent_dim, kernel_size=4, stride=4)
        
        # Shared stem
        self.shared_net = nn.Sequential(
            nn.Conv1d(540 + latent_dim + 128, 512, kernel_size=3, padding=1), nn.GELU(),
            nn.Conv1d(512, 512, kernel_size=3, padding=1), nn.GELU(),
        )
        
        # Separate heads for parts and features
        self.heads = nn.ModuleDict({
            "body_pos": nn.Conv1d(512, 99, kernel_size=3, padding=1),
            "body_vel": nn.Conv1d(512, 99, kernel_size=3, padding=1),
            "face_pos": nn.Conv1d(512, 45, kernel_size=3, padding=1),
            "face_vel": nn.Conv1d(512, 45, kernel_size=3, padding=1),
            "lhand_pos": nn.Conv1d(512, 63, kernel_size=3, padding=1),
            "lhand_vel": nn.Conv1d(512, 63, kernel_size=3, padding=1),
            "rhand_pos": nn.Conv1d(512, 63, kernel_size=3, padding=1),
            "rhand_vel": nn.Conv1d(512, 63, kernel_size=3, padding=1)
        })

    def forward(self, noisy_flat, z, t):
        x = noisy_flat.transpose(1, 2) # [B, 540, T]
        t_emb = self.time_mlp(t.float()).unsqueeze(-1).expand(-1, -1, x.shape[-1])
        z_up = self.upsample_z(z.transpose(1, 2))
        
        if z_up.shape[-1] != x.shape[-1]:
            z_up = F.interpolate(z_up, size=x.shape[-1], mode='linear', align_corners=False)
            
        shared_out = self.shared_net(torch.cat([x, z_up, t_emb], dim=1)) # [B, 512, T]
        
        pred = {}
        for key, head in self.heads.items():
            pred[key] = head(shared_out).transpose(1, 2)
        return pred

def apply_masking_and_noise(clean_dict, device="cpu"):
    B, T, _ = clean_dict["body_pos"].shape
    masked_dict = {}
    masks = {}
    mask_type = random.random()
    
    for key, val in clean_dict.items():
        mask = torch.zeros_like(val)
        if mask_type < 0.3:
            drop_mask = (torch.rand_like(val) < 0.15).float()
            mask = drop_mask
        elif mask_type < 0.6:
            span_len = int(T * 0.2)
            start = random.randint(0, max(0, T - span_len))
            mask[:, start:start+span_len, :] = 1.0
        elif mask_type < 0.9:
            if "lhand" in key and random.random() < 0.5: mask[:] = 1.0
            elif "rhand" in key and random.random() < 0.5: mask[:] = 1.0
            elif "face" in key and random.random() < 0.2: mask[:] = 1.0
        
        noise = torch.randn_like(val)
        t = torch.randint(0, 1000, (B, 1, 1), device=device)
        alpha_t = (1 - t / 1000)
        
        noisy = alpha_t * val + (1 - alpha_t) * noise
        noisy[mask == 1.0] = noise[mask == 1.0]
        
        masked_dict[key] = noisy
        masks[key] = mask
        
    return masked_dict, masks, t

def compute_phase1_loss(pred_dict, clean_dict, masks_dict, z):
    L_pos, L_vel, L_full = 0.0, 0.0, 0.0
    for key in clean_dict.keys():
        pred, clean, mask = pred_dict[key], clean_dict[key], masks_dict[key]
        mse = F.mse_loss(pred, clean, reduction='none')
        masked_mse = (mse * mask).sum() / (mask.sum() + 1e-8)
        full_mse = mse.mean()
        if "pos" in key:
            L_pos += masked_mse
        else:
            L_vel += masked_mse
        L_full += full_mse
    L_latent = torch.mean((z[:, 1:, :] - z[:, :-1, :])**2)
    return 1.0 * L_pos + 1.0 * L_vel + 0.1 * L_full + 0.01 * L_latent


## Dataset — streaming with on-the-fly feature engineering

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset

FACE_LANDMARK_IDXS = [70, 105, 336, 300, 33, 133, 362, 263, 4, 61, 291, 13, 14, 17, 0]
_POSE_SLICE = slice(0, 33)
_FACE_SLICE = slice(33, 501)
_LHAND_SLICE = slice(501, 522)
_RHAND_SLICE = slice(522, 543)
T_WINDOW, T_STRIDE = 60, 30

def engineer_features_multistream(raw: np.ndarray):
    T = raw.shape[0]
    if T < 2: return None

    pose = raw[:, _POSE_SLICE, :]
    face = raw[:, _FACE_SLICE, :][:, FACE_LANDMARK_IDXS, :]
    lhand = raw[:, _LHAND_SLICE, :]
    rhand = raw[:, _RHAND_SLICE, :]

    shoulder_center = (pose[:, 11, :] + pose[:, 12, :]) / 2.0
    pose_norm = pose - shoulder_center[:, None, :]
    
    face_center = face.mean(axis=1, keepdims=True)
    face_norm = face - face_center
    
    lhand_norm = lhand - lhand[:, 0:1, :]
    rhand_norm = rhand - rhand[:, 0:1, :]

    def calc_vel(x):
        vel = np.zeros_like(x)
        vel[1:] = x[1:] - x[:-1]
        return vel

    return {
        "body_pos": torch.from_numpy(pose_norm.reshape(T, -1).astype(np.float32)),
        "face_pos": torch.from_numpy(face_norm.reshape(T, -1).astype(np.float32)),
        "lhand_pos": torch.from_numpy(lhand_norm.reshape(T, -1).astype(np.float32)),
        "rhand_pos": torch.from_numpy(rhand_norm.reshape(T, -1).astype(np.float32)),
        "body_vel": torch.from_numpy(calc_vel(pose_norm).reshape(T, -1).astype(np.float32)),
        "face_vel": torch.from_numpy(calc_vel(face_norm).reshape(T, -1).astype(np.float32)),
        "lhand_vel": torch.from_numpy(calc_vel(lhand_norm).reshape(T, -1).astype(np.float32)),
        "rhand_vel": torch.from_numpy(calc_vel(rhand_norm).reshape(T, -1).astype(np.float32))
    }

def sliding_windows_dict(feat_dict, window=T_WINDOW, stride=T_STRIDE):
    T = feat_dict["body_pos"].shape[0]
    if T < window:
        yield {k: F.pad(v, (0, 0, 0, window - T)) for k, v in feat_dict.items()}
    else:
        start = 0
        while start + window <= T:
            yield {k: v[start:start + window] for k, v in feat_dict.items()}
            start += stride

class RealSignLanguageDataset(IterableDataset):
    def __init__(self, split="train", max_samples=None, repo_id="bdanko/how2sign-landmarks-front-raw-parquet"):
        self.split, self.max_samples, self.repo_id = split, max_samples, repo_id

    def __iter__(self):
        from datasets import load_dataset
        ds = load_dataset(self.repo_id, split=self.split, streaming=True)
        count = 0
        for sample in ds:
            if self.max_samples and count >= self.max_samples:
                break
            raw = np.frombuffer(sample["features"], dtype=np.float32).reshape(sample["shape"])
            feat_dict = engineer_features_multistream(raw)
            if feat_dict is None:
                continue
            for chunk in sliding_windows_dict(feat_dict):
                yield chunk, sample.get("sentence", "")
            count += 1


## ⚙️ Configuration
Adjust these before running the training cell.


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────────
REPO_ID             = "bdanko/continuous-sign-language-translation"
SPLIT               = "train"
DEBUG_MAX_SAMPLES   = 100
FULL_MAX_SAMPLES    = None
MAX_SAMPLES         = DEBUG_MAX_SAMPLES  # Switch to FULL_MAX_SAMPLES for real run
BATCH_SIZE          = 32
LR                  = 1e-4
EPOCHS              = 3
CKPT_DIR            = "/tmp/phase1_ckpt"


## Training loop

In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from tqdm.auto import tqdm

os.makedirs(CKPT_DIR, exist_ok=True)

encoder = MultiStreamSemanticEncoder().to(device)
decoder = StructuredDiffusionDecoder().to(device)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LR)

dataset = RealSignLanguageDataset(split=SPLIT, max_samples=MAX_SAMPLES)

for epoch in range(EPOCHS):
    total_loss, count = 0.0, 0
    batch_buf = []

    for features, _ in dataset:
        batch_buf.append(features)
        if len(batch_buf) < BATCH_SIZE:
            continue

        batch = {k: torch.stack([d[k] for d in batch_buf]).to(device) for k in batch_buf[0].keys()}
        batch_buf = []

        # Apply masks and noise FIRST
        noisy_dict, masks, t = apply_masking_and_noise(batch, device)
        
        # Flatten noisy dict for decoder input
        noisy_flat = torch.cat([noisy_dict[k] for k in [
            "body_pos", "body_vel", "face_pos", "face_vel", 
            "lhand_pos", "lhand_vel", "rhand_pos", "rhand_vel"
        ]], dim=-1)

        # Encode from NOISY to force robust encoding
        z = encoder(noisy_dict)  # [B, T', D]

        # Decode
        pred_dict = decoder(noisy_flat, z, t)
        
        # Compute losses separately
        loss = compute_phase1_loss(pred_dict, batch, masks, z)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        count += 1

    avg = total_loss / max(count, 1)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg_loss={avg:.4f}")

print("Training complete.")


## Save & upload to Hugging Face

In [ ]:
import torch
from huggingface_hub import HfApi, create_repo, upload_folder

torch.save(encoder.state_dict(), f"{CKPT_DIR}/semantic_encoder.pth")
torch.save(decoder.state_dict(), f"{CKPT_DIR}/diffusion_decoder.pth")
print(f"Checkpoints saved to {CKPT_DIR}/")

api = HfApi()
create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True)
upload_folder(
    folder_path=CKPT_DIR,
    repo_id=REPO_ID,
    token=HF_TOKEN,
    commit_message=f"Phase 1 checkpoint — {EPOCHS} epochs, {MAX_SAMPLES} clips",
)
print(f"Uploaded to https://huggingface.co/{REPO_ID} ✓")
